# AcuDock Pro - Interactive Molecular Docking with CNN Rescoring

**Approach 2** from the AcuDock project: a widget-driven, interactive molecular docking
notebook combining **AutoDock Vina** with **Gnina CNN rescoring** for consensus scoring.

## Getting Started

1. Run the **first code cell** below to install packages (takes ~2-3 minutes)
2. The runtime will **automatically restart** -- this is normal
3. After restart, **skip the install cell** and run from the imports cell onward

## Why CNN Rescoring?

AutoDock Vina uses a traditional empirical scoring function that achieves ~58% redocking
success (RMSD < 2 A). Gnina applies a **convolutional neural network** trained on the
PDBBind dataset to re-evaluate binding poses, boosting redocking success to **~73%** --
a significant improvement in pose prediction accuracy.

## Features

- **Interactive widgets** for all parameters (no code editing required)
- **Three scoring modes:** Vina only, Vina + Gnina CNN, or Consensus (z-score weighted)
- **3D visualization** with py3Dmol (protein cartoon + ligand sticks)
- **Multi-pose overlay** to compare binding orientations
- **Batch screening** mode for compound libraries
- **Interaction fingerprints** via ProLIF (when available)

## Pipeline

```
PDB ID --> PDBFixer --> PDBQT (receptor)
SMILES --> RDKit 3D --> Meeko --> PDBQT (ligand)
         |                             |
         v                             v
     Vina Docking ----> Gnina CNN Rescore ----> Consensus Rank
         |                                          |
         +---------> 3D Visualization <-------------+
```

**License:** MIT | **Platform:** Google Colab | **Author:** AcuDock Project

In [ ]:
# Install all dependencies
!pip install vina meeko rdkit-pypi prody py3Dmol prolif openbabel-wheel pdbfixer pandas numpy scipy ipywidgets

# Download Gnina binary for CNN rescoring
!wget -q https://github.com/gnina/gnina/releases/latest/download/gnina -O /usr/local/bin/gnina && chmod +x /usr/local/bin/gnina

# Restart runtime so C-extension packages (vina, rdkit, openbabel) are loadable.
# After restart, skip this cell and continue from the next one.
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── Imports and Setup ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
import meeko
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, HTML, Image

# Import AcuDock shared utilities
import acudock_utils as utils

# Working directory
WORK_DIR = '/content/acudock_pro'
os.makedirs(WORK_DIR, exist_ok=True)

print('AcuDock Pro loaded successfully.')
print(f'Working directory: {WORK_DIR}')

## Configuration Panel

Use the interactive widgets below to configure all docking parameters.
No code editing is required -- just adjust the controls and run the cells.

In [ ]:
# ── Widget Definitions ───────────────────────────────────────────────────────

# Target protein
pdb_input = widgets.Text(
    value='1HSG',
    description='PDB ID:',
    placeholder='e.g. 1HSG, 4LDE, 6LU7',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Ligand SMILES
smiles_input = widgets.Textarea(
    value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
    description='SMILES:',
    placeholder='Enter ligand SMILES string',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px', height='60px')
)

# Ligand name
ligand_name = widgets.Text(
    value='Indinavir',
    description='Ligand Name:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Docking parameters
exhaustiveness_slider = widgets.IntSlider(
    value=32, min=8, max=128, step=8,
    description='Exhaustiveness:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

n_poses_slider = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description='Num Poses:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

box_size_slider = widgets.IntSlider(
    value=20, min=15, max=40, step=5,
    description='Box Size (A):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Scoring engine
engine_dropdown = widgets.Dropdown(
    options=['Vina', 'Vina + Gnina CNN', 'Consensus'],
    value='Vina',
    description='Engine:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Active site residues
active_site_residues = widgets.Text(
    value='23,24,25,26,27,28,29,30',
    description='Active Site:',
    placeholder='Comma-separated residue IDs',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px')
)

# Layout
panel = widgets.VBox([
    widgets.HTML('<h3>Target Protein</h3>'),
    pdb_input,
    active_site_residues,
    widgets.HTML('<h3>Ligand</h3>'),
    ligand_name,
    smiles_input,
    widgets.HTML('<h3>Docking Parameters</h3>'),
    engine_dropdown,
    exhaustiveness_slider,
    n_poses_slider,
    box_size_slider,
])

display(panel)

## Protein Preparation

Fetches the protein structure from the RCSB PDB, then applies PDBFixer to:
- Fill missing residues and atoms
- Replace non-standard residues
- Remove heterogens (water, ions, co-crystallized ligands)
- Add hydrogens at physiological pH (7.4)

The prepared PDB is then converted to PDBQT format for docking.

In [ ]:
# ── Prepare Protein ──────────────────────────────────────────────────────────
pdb_id = pdb_input.value.strip().upper()
print(f'Fetching and preparing protein: {pdb_id}')

protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
print(f'  Prepared PDB: {protein_pdb}')

receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)
print(f'  Receptor PDBQT: {receptor_pdbqt}')
print('Protein preparation complete.')

# 3D visualization of the prepared protein
with open(protein_pdb, 'r') as f:
    protein_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(protein_data, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum', 'opacity': 0.9}})
view.setBackgroundColor('white')
view.zoomTo()
view.show()

In [ ]:
# ── Prepare Ligand ───────────────────────────────────────────────────────────
smi = smiles_input.value.strip()
lig_name = ligand_name.value.strip() or 'ligand'
print(f'Preparing ligand: {lig_name}')
print(f'  SMILES: {smi}')

ligand_pdbqt, lig_mol = utils.prepare_ligand(smi, name=lig_name, output_dir=WORK_DIR)
print(f'  Ligand PDBQT: {ligand_pdbqt}')

# Display 2D structure
mol_2d = Chem.MolFromSmiles(smi)
img = Draw.MolToImage(mol_2d, size=(400, 300))
display(img)

# Print molecular properties
props = utils.get_ligand_properties(smi)
print('\nMolecular Properties:')
print(f'  Molecular Weight:    {props["MW"]} Da')
print(f'  LogP:                {props["LogP"]}')
print(f'  H-Bond Donors:       {props["HBD"]}')
print(f'  H-Bond Acceptors:    {props["HBA"]}')
print(f'  Rotatable Bonds:     {props["RotBonds"]}')
print(f'  TPSA:                {props["TPSA"]} A^2')

# Lipinski Rule of Five check
violations = sum([
    props['MW'] > 500,
    props['LogP'] > 5,
    props['HBD'] > 5,
    props['HBA'] > 10
])
print(f'\n  Lipinski Violations: {violations}/4', '(PASS)' if violations <= 1 else '(FAIL)')

## Docking Execution

Three scoring engines are available via the dropdown above:

| Engine | Description | Speed | Accuracy |
|--------|-------------|-------|----------|
| **Vina** | Classical empirical scoring function | Fast | ~58% redocking |
| **Vina + Gnina CNN** | Vina docking + CNN-based rescoring | Moderate | ~73% redocking |
| **Consensus** | Z-score weighted combination of both | Moderate | Best overall |

The search box is centered on the active-site residues specified in the configuration panel.

In [ ]:
# ── Define Search Box ────────────────────────────────────────────────────────
residue_str = active_site_residues.value.strip()
if residue_str:
    residue_ids = [int(r.strip()) for r in residue_str.split(',') if r.strip()]
else:
    residue_ids = None

center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residue_ids)
box = [box_size_slider.value] * 3

print(f'Search box center: [{center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}] A')
print(f'Search box size:   {box} A')
if residue_ids:
    print(f'Based on residues: {residue_ids}')
else:
    print('Based on: whole protein centroid')

In [ ]:
# ── Run Docking ──────────────────────────────────────────────────────────────
engine = engine_dropdown.value
exh = exhaustiveness_slider.value
n_poses = n_poses_slider.value

print(f'Engine:          {engine}')
print(f'Exhaustiveness:  {exh}')
print(f'Max poses:       {n_poses}')
print('-' * 50)

# Step 1: Always run Vina docking
print('Running AutoDock Vina...')
vina_obj, energies, poses_path = utils.run_vina(
    receptor_pdbqt, ligand_pdbqt,
    center=center, box_size=box,
    exhaustiveness=exh, n_poses=n_poses
)
print(f'  Vina complete: {len(energies)} poses generated')
print(f'  Best Vina score: {energies[0][0]:.2f} kcal/mol')

# Step 2: Gnina CNN rescoring (if requested)
gnina_scores = None
if engine in ['Vina + Gnina CNN', 'Consensus']:
    print('\nRunning Gnina CNN rescoring...')
    gnina_raw = utils.run_gnina_rescore(receptor_pdbqt, poses_path, output_dir=WORK_DIR)
    if gnina_raw is not None:
        # Extract CNN affinity values
        gnina_scores = [s['value'] for s in gnina_raw if s['metric'] == 'CNNaffinity']
        if gnina_scores:
            print(f'  Gnina CNN rescoring complete: {len(gnina_scores)} scores')
        else:
            print('  Warning: Could not parse CNN affinity scores')
            gnina_scores = None
    else:
        print('  Warning: Gnina unavailable, using Vina scores only')

# Step 3: Build results
results_df = utils.energies_to_dataframe(energies, ligand_name=lig_name)

if engine == 'Consensus' and gnina_scores is not None:
    consensus_df = utils.consensus_score(energies, gnina_scores, alpha=0.5)
    print('\nConsensus scoring complete.')
else:
    consensus_df = None

print('\nDocking complete.')

## Results & Visualization

Explore the docking results below: score tables, 3D pose visualization,
multi-pose overlays, and interaction analysis.

In [ ]:
# ── Score Table ──────────────────────────────────────────────────────────────
print('=' * 60)
print(f'  DOCKING RESULTS: {lig_name} in {pdb_id}')
print('=' * 60)

if consensus_df is not None:
    print('\nConsensus Ranking (Vina + Gnina CNN):')
    display(consensus_df)
else:
    print('\nVina Docking Results:')
    display(results_df)

# Interpret top pose
top_score = energies[0][0]
interpretation = utils.score_interpretation(top_score)
print(f'\nTop Pose Score: {top_score:.2f} kcal/mol')
print(f'Interpretation: {interpretation}')
print(f'Estimated Kd:   {results_df["Est_Kd_uM"].iloc[0]:.4f} uM')

In [ ]:
# ── 3D Visualization - Selectable Pose ───────────────────────────────────────
num_poses = len(energies)

pose_selector = widgets.Dropdown(
    options=[(f'Pose {i+1} ({energies[i][0]:.2f} kcal/mol)', i) for i in range(num_poses)],
    value=0,
    description='Select Pose:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

def show_pose(pose_index):
    """Render the selected pose in 3D."""
    print(f'Showing pose {pose_index + 1}: {energies[pose_index][0]:.2f} kcal/mol')
    view = utils.visualize_pose(protein_pdb, poses_path, pose_index=pose_index)
    view.show()

out = widgets.interactive_output(show_pose, {'pose_index': pose_selector})
display(pose_selector, out)

In [ ]:
# ── Multi-Pose Overlay (Top 3) ──────────────────────────────────────────────
n_overlay = min(3, len(energies))
print(f'Overlaying top {n_overlay} poses on the protein structure:')
for i in range(n_overlay):
    color = ['green', 'cyan', 'magenta'][i]
    print(f'  Pose {i+1} ({color}): {energies[i][0]:.2f} kcal/mol')

multi_view = utils.visualize_multi_poses(protein_pdb, poses_path, n_poses=n_overlay)
multi_view.show()

In [ ]:
# ── Interaction Analysis (ProLIF) ────────────────────────────────────────────
try:
    import prolif
    import MDAnalysis as mda

    print('Generating protein-ligand interaction fingerprints with ProLIF...')

    # Load protein and top pose
    prot = mda.Universe(protein_pdb)
    prot = prolif.Molecule.from_mda(prot)

    # Read top pose from PDBQT
    with open(poses_path, 'r') as f:
        poses_data = f.read()
    models = poses_data.split('MODEL')
    if len(models) > 1:
        top_pose_data = 'MODEL' + models[1].split('ENDMDL')[0] + 'ENDMDL'
    else:
        top_pose_data = poses_data

    # Write top pose to temporary PDB
    top_pose_path = os.path.join(WORK_DIR, 'top_pose.pdb')
    with open(top_pose_path, 'w') as f:
        f.write(top_pose_data)

    lig_u = mda.Universe(top_pose_path)
    lig = prolif.Molecule.from_mda(lig_u)

    # Generate fingerprint
    fp = prolif.Fingerprint()
    fp.run_from_iterable([lig], prot)

    ifp_df = fp.to_dataframe()
    print('\nInteraction Fingerprint:')
    display(ifp_df)

except ImportError:
    print('ProLIF not available for interaction analysis.')
    print('This is optional -- docking results are still valid.')
    print('To enable, install: !pip install prolif MDAnalysis')
except Exception as e:
    print(f'Interaction analysis encountered an error: {e}')
    print('This is optional -- docking results are still valid.')

## Batch Screening

Screen a small library of compounds against the same target.
Batch mode uses reduced exhaustiveness (8) for faster throughput.
The results are ranked by docking score.

In [ ]:
# ── Batch Compounds Library ──────────────────────────────────────────────────
compound_library = [
    ('Aspirin',       'CC(=O)OC1=CC=CC=C1C(=O)O'),
    ('Ibuprofen',     'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O'),
    ('Caffeine',      'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'),
    ('Acetaminophen', 'CC(=O)NC1=CC=C(O)C=C1'),
    ('Naproxen',      'COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O'),
    ('Metformin',     'CN(C)C(=N)NC(=N)N'),
    ('Atorvastatin',  'CC(C)C1=C(C(=CC=C1)C2=CC=C(C=C2)F)C3=CC=CC=C3C(=O)NC(CC[C@@H](O)C[C@@H](O)CC(=O)O)C(C)C'),
    ('Omeprazole',    'CC1=CN=C(C(=C1OC)C)CS(=O)C2=NC3=CC=CC=C3N2'),
]

print(f'Compound library: {len(compound_library)} compounds')
for name, smi in compound_library:
    props = utils.get_ligand_properties(smi)
    print(f'  {name:15s}  MW={props["MW"]:6.1f}  LogP={props["LogP"]:5.2f}')

In [ ]:
# ── Run Batch Docking ────────────────────────────────────────────────────────
batch_results = []
batch_exhaustiveness = 8  # Lower for speed in batch mode

print(f'Batch docking {len(compound_library)} compounds against {pdb_id}')
print(f'Exhaustiveness: {batch_exhaustiveness} (reduced for batch speed)')
print('=' * 60)

for idx, (name, smi) in enumerate(compound_library):
    print(f'\n[{idx+1}/{len(compound_library)}] Docking {name}...')
    try:
        # Prepare ligand
        lig_pdbqt, _ = utils.prepare_ligand(smi, name=name, output_dir=WORK_DIR)

        # Run Vina
        _, lig_energies, _ = utils.run_vina(
            receptor_pdbqt, lig_pdbqt,
            center=center, box_size=box,
            exhaustiveness=batch_exhaustiveness, n_poses=5
        )

        best_score = lig_energies[0][0]
        lig_props = utils.get_ligand_properties(smi)
        est_kd = np.exp(best_score / (1.987e-3 * 298.15)) * 1e6

        batch_results.append({
            'Name': name,
            'SMILES': smi,
            'Score_kcal_mol': best_score,
            'Est_Kd_uM': round(est_kd, 4),
            'MW': lig_props['MW'],
            'LogP': lig_props['LogP'],
            'Interpretation': utils.score_interpretation(best_score),
        })
        print(f'  Score: {best_score:.2f} kcal/mol -- {utils.score_interpretation(best_score)}')

    except Exception as e:
        print(f'  ERROR: {e}')
        batch_results.append({
            'Name': name,
            'SMILES': smi,
            'Score_kcal_mol': None,
            'Est_Kd_uM': None,
            'MW': None,
            'LogP': None,
            'Interpretation': 'Failed',
        })

print('\n' + '=' * 60)
print('Batch docking complete.')

In [ ]:
# ── Batch Results ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

batch_df = pd.DataFrame(batch_results)
batch_df = batch_df.sort_values('Score_kcal_mol', ascending=True).reset_index(drop=True)
batch_df.index = batch_df.index + 1
batch_df.index.name = 'Rank'

print('Batch Screening Results (sorted by score):')
display(batch_df[['Name', 'Score_kcal_mol', 'Est_Kd_uM', 'MW', 'LogP', 'Interpretation']])

# Bar chart of docking scores
valid = batch_df.dropna(subset=['Score_kcal_mol'])

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
          for s in valid['Score_kcal_mol']]
ax.barh(valid['Name'], valid['Score_kcal_mol'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Docking Score (kcal/mol)', fontsize=12)
ax.set_title(f'Batch Screening: {pdb_id}', fontsize=14)
ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5, label='Moderate threshold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Export Results ────────────────────────────────────────────────────────────
# Save single ligand results
single_csv = os.path.join(WORK_DIR, f'{lig_name}_{pdb_id}_results.csv')
results_df.to_csv(single_csv, index=False)
print(f'Single docking results saved: {single_csv}')

# Save batch results
batch_csv = os.path.join(WORK_DIR, f'batch_{pdb_id}_results.csv')
batch_df.to_csv(batch_csv, index=True)
print(f'Batch screening results saved: {batch_csv}')

# Save consensus results if available
if consensus_df is not None:
    consensus_csv = os.path.join(WORK_DIR, f'{lig_name}_{pdb_id}_consensus.csv')
    consensus_df.to_csv(consensus_csv, index=True)
    print(f'Consensus results saved: {consensus_csv}')

# Offer download in Colab
try:
    from google.colab import files
    print('\nDownloading results...')
    files.download(single_csv)
    files.download(batch_csv)
    if consensus_df is not None:
        files.download(consensus_csv)
except ImportError:
    print('\nNot running in Google Colab -- files saved to working directory.')
    print(f'Directory: {WORK_DIR}')

## Summary

### AcuDock Pro Features

| Feature | Description |
|---------|-------------|
| **Interactive Widgets** | All parameters adjustable without code editing |
| **Protein Preparation** | PDBFixer: missing atoms, hydrogens at pH 7.4 |
| **Ligand Preparation** | RDKit ETKDGv3 + MMFF optimization + Meeko PDBQT |
| **Vina Docking** | AutoDock Vina with configurable exhaustiveness |
| **Gnina CNN Rescoring** | CNN-based rescoring (58% -> 73% redocking accuracy) |
| **Consensus Scoring** | Z-score weighted Vina + Gnina combination |
| **3D Visualization** | py3Dmol: protein cartoon + ligand sticks + surface |
| **Multi-Pose Overlay** | Compare top poses side by side |
| **Interaction Analysis** | ProLIF fingerprints (when available) |
| **Batch Screening** | Screen compound libraries with ranked results |
| **Export** | CSV download of all results |

### Score Interpretation Guide

| Score (kcal/mol) | Binding | Approximate Kd |
|:----------------:|---------|----------------|
| > -5 | Very weak / none | > 100 uM |
| -5 to -6 | Weak | mM range |
| -6 to -7 | Moderate-weak | ~100 uM |
| -7 to -8 | Moderate | ~10 uM |
| -8 to -9 | Good | sub-uM |
| -9 to -10 | Strong | ~50 nM |
| < -10 | Very strong | < 50 nM (verify) |

### Next Steps

- **Validate:** Redock a co-crystallized ligand and check RMSD < 2 A
- **Scale up:** Use **AcuDock Scout** for active-learning-driven virtual screening of large libraries (10k+ compounds)
- **Publish:** Use Consensus mode (Vina + Gnina) for publication-quality results
- **Extend:** Add MD refinement for top hits (coming in AcuDock Suite)

---
*AcuDock Pro -- Interactive Molecular Docking with CNN Rescoring*
*MIT License | Built for Google Colab*